In [1]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [2]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [3]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [4]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [5]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [6]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [7]:
rec = answers[0]
rec

{'question': 'I just found this course, is it too late to join?',
 'answer_llm': 'Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [8]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)
print(prompt)

Question:
I just found this course, is it too late to join?

Original Answer (ground truth):
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

AI Answer:
Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still being accepted.


In [9]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the ground truth meaning: it says it is still possible to join, and that receiving a certificate requires submitting the project before submissions close. This is semantically equivalent.', score='good')

In [10]:
calc_price(usage)

{'input_cost': 0.0002175, 'output_cost': 0.0002385, 'total_cost': 0.000456}

In [11]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [12]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer conveys the same meaning as the ground truth: it says it is still possible to join, but a certificate requires submitting the project while submissions are open. This matches the original answer semantically.', score='good')

In [13]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [14]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/590 [00:00<?, ?it/s]

In [15]:
results[10]

({'question': 'Where do I watch the live office hours or workshop stream if I’m a student?',
  'document': '489dd1c9d9',
  'score': 'good',
  'reasoning': 'The AI answer matches the ground truth: it says students watch via the DataTalksClub YouTube channel, that the stream URL is posted in the announcements channel on Telegram and Slack before it starts, and that the Slido link is pinned in chat when live. It omits the note not to post questions in chat, but that is extra guidance rather than the core answer.'},
 ResponseUsage(input_tokens=409, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=93, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=502))

In [16]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [17]:
calc_total_price(usages)

0.4195409999999998

In [18]:
df_eval = pd.DataFrame(evaluations)

In [19]:
df_eval.head()

,question,document,score,reasoning
0,"I just found this course, is it too late to join?",74eb249bbf,good,The AI answer preserves the key meaning of the...
1,Can I still sign up even if the course already...,74eb249bbf,bad,"The ground truth says yes, late signup is allo..."
2,"If I join now, can I still get a certificate s...",74eb249bbf,good,The AI answer preserves the key point: joining...
3,What do I need to do to be eligible for the ce...,74eb249bbf,good,The AI answer preserves the core requirement f...
4,"Are late joiners allowed, and does that affect...",74eb249bbf,good,The AI answer preserves the core idea that lat...


In [20]:
df_eval.score.value_counts()

score
good    560
bad      30
Name: count, dtype: int64

In [21]:
df_eval.score.value_counts(normalize=True)

score
good    0.949153
bad     0.050847
Name: proportion, dtype: float64

In [22]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
1,Can I still sign up even if the course already...,74eb249bbf,bad,"The ground truth says yes, late signup is allo..."
26,If I study the material self-paced and build t...,69d122f12e,bad,The AI answer captures the key requirement tha...
31,Do I have to do all the homework to get the co...,9f689c185f,bad,The AI answer captures the core point that hom...
34,What do I need to pass in order to receive the...,9f689c185f,bad,The ground truth says the certificate requires...
40,When is the next llm-zoomcamp session starting?,bd31146b0e,bad,The ground truth specifies the next llm-zoomca...


In [23]:
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)
